In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_train.xlsx
/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_validation.xlsx
/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_unlabeled.xlsx


In [2]:
df_train = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_train.xlsx')
df_unlabeled = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_unlabeled.xlsx')
df_validation = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_validation.xlsx')


In [3]:
print("shape\n",df_train.shape)
print("describtion\n",df_train.describe())
print("nulls")
print(df_train.isnull().sum())

shape
 (1971, 9)
describtion
           review_id  star_rating
count   1971.000000  1971.000000
mean    5047.205479     3.341451
std     2800.399699     1.821920
min       16.000000     1.000000
25%     2646.000000     1.000000
50%     5122.000000     5.000000
75%     7377.500000     5.000000
max    10010.000000     5.000000
nulls
review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64


In [65]:
df_train

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"
...,...,...,...,...,...,...,...,...,...
1966,6530,بصراحه كنت بحبو جدا جدا والناس كانت محترمه بس ...,1,2026-03-10 00:00:00,Careem,transport,play_store,"[""service"", ""delivery"", ""price"", ""app_experien...","{""service"": ""positive"", ""delivery"": ""negative""..."
1967,4612,المطعم قعدته حلوه وفى زحليقه ومرجيحه للاطفال\n...,3,قبل 3 أشهر,Samakmak Restaurant - 6th of October Branch,مطعم مأكولات بحرية,google_maps,"[""ambiance"", ""food"", ""price"", ""service""]","{""ambiance"": ""positive"", ""food"": ""neutral"", ""p..."
1968,4769,The worst experience at the HUB. Place is dirt...,1,قبل سنتين,ال ديفينو بيتزيريا,مطعم بيتزا,google_maps,"[""cleanliness"", ""food""]","{""cleanliness"": ""negative"", ""food"": ""negative""}"
1969,5366,الكوفي في فندق الهيلتون رمسيس...\n\nتم اضافه س...,1,قبل 4 سنوات,جاردن كورت كافيه,مقهى,google_maps,"[""service""]","{""service"": ""negative""}"


In [4]:
print(df_train.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [5]:
df_train.columns = (
    df_train.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [6]:
print(df_train.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [7]:
#arabic normalize
def normalize_arabic(text):
    text = str(text)
    
    # normalize letters
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    
    # remove brackets like (2)
    text = re.sub(r'\(.*?\)', '', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    return text

In [8]:
df_train['date']

0       2026-03-08 00:00:00
1             قبل يومين (2)
2                   قبل شهر
3                   قبل شهر
4                   قبل سنة
               ...         
1966    2026-03-10 00:00:00
1967             قبل 3 أشهر
1968              قبل سنتين
1969            قبل 4 سنوات
1970                قبل شهر
Name: date, Length: 1971, dtype: object

In [9]:
# import pandas as pd
# import re
# from datetime import datetime

# #refrence date
# reference_date = pd.Timestamp.today()  # or pd.Timestamp.today()



# #tokenization
# def tokenize(text):
#     return text.split()

# #semantic vocabulary
# time_units = {
#     # days
#     "يوم": 1,
#     "يومين": 2,
#     "ايام": 1,
    
#     # months
#     "شهر": 30,
#     "اشهر": 30,
    
#     # years
#     "سنه": 365,
#     "سنين": 365,
#     "سنتين": 2 * 365,
#     "سنوات": 365
# }

# # extract number helper
# def extract_number(tokens):
#     for t in tokens:
#         if t.isdigit():
#             return int(t)
#     return None

# # NLP-based parsing
# def parse_date_nlp(text):
#     text = normalize_arabic(text)

#     # absolute date ----
#     try:
#         dt = pd.to_datetime(text)
#         return (reference_date - dt).days
#     except:
#         pass

#     # relative Arabic ----
#     tokens = tokenize(text)

#     number = extract_number(tokens)
#     unit_value = None

#     for t in tokens:
#         if t in time_units:
#             unit_value = time_units[t]
#             break

#     if unit_value is not None:
#         if number is None:
#             number = 1
#         return number * unit_value

#     # fallback 
#     return None

# # apply on dataset
# df_train['days_ago'] = df_train['date'].apply(parse_date_nlp)

# # handle missing
# df_train['days_ago'] = df_train['days_ago'].fillna(df_train['days_ago'].median())



In [10]:
import pandas as pd
import re
from datetime import timedelta

# reference date
reference_date = pd.Timestamp.today()  # or pd.Timestamp.today()

# tokenization
def tokenize(text):
    return text.split()

# time vocab (days)
time_units = {
    "يوم": 1,
    "يومين": 2,
    "ايام": 1,
    
    "شهر": 30,
    "اشهر": 30,
    
    "سنه": 365,
    "سنين": 365,
    "سنتين": 2 * 365,
    "سنوات": 365
}

# extract number
def extract_number(tokens):
    for t in tokens:
        if t.isdigit():
            return int(t)
    return None

# convert to DATETIME
def parse_to_datetime(text):
    text = normalize_arabic(text)

    # absolute date ----
    try:
        return pd.to_datetime(text)
    except:
        pass

    # relative Arabic ----
    tokens = tokenize(text)
    number = extract_number(tokens)
    unit_value = None

    for t in tokens:
        if t in time_units:
            unit_value = time_units[t]
            break

    if unit_value is not None:
        if number is None:
            number = 1
        
        days = number * unit_value
        return reference_date - timedelta(days=days)

    return pd.NaT

# apply
df_train['date_parsed'] = df_train['date'].apply(parse_to_datetime)

# extract features
df_train['year'] = df_train['date_parsed'].dt.year
df_train['month'] = df_train['date_parsed'].dt.month
df_train['day_of_week'] = df_train['date_parsed'].dt.dayofweek  # 0=Mon , 6=Sunday and so on

# handle missing
df_train['year'] = df_train['year'].fillna(df_train['year'].median())
df_train['month'] = df_train['month'].fillna(df_train['month'].median())
df_train['day_of_week'] = df_train['day_of_week'].fillna(df_train['day_of_week'].median())

In [11]:
df_train['date_parsed']

0      2026-03-08 00:00:00.000000
1      2026-04-22 11:11:24.557063
2      2026-03-25 11:11:24.557063
3      2026-03-25 11:11:24.557063
4      2025-04-24 11:11:24.557063
                  ...            
1966   2026-03-10 00:00:00.000000
1967   2026-01-24 11:11:24.557063
1968   2024-04-24 11:11:24.557063
1969   2022-04-25 11:11:24.557063
1970   2026-03-25 11:11:24.557063
Name: date_parsed, Length: 1971, dtype: datetime64[ns]

In [12]:
print(type(df_train))

<class 'pandas.core.frame.DataFrame'>


In [13]:
def identify_outliers(column_values):
  plt.figure(figsize=(8, 4))
  sns.boxplot(x=column_values, color='lightgreen')
  plt.title('Box Plot to Identify Outliers and Symmetry')
  plt.show()

In [14]:
df_train.drop_duplicates(inplace=True)

In [15]:
df_train.shape

(1971, 13)

In [16]:
df_train['aspect_sentiments'].value_counts

<bound method IndexOpsMixin.value_counts of 0       {"app_experience": "negative", "delivery": "ne...
1       {"cleanliness": "positive", "ambiance": "posit...
2       {"service": "negative", "delivery": "negative"...
3                                 {"general": "positive"}
4               {"food": "positive", "price": "positive"}
                              ...                        
1966    {"service": "positive", "delivery": "negative"...
1967    {"ambiance": "positive", "food": "neutral", "p...
1968      {"cleanliness": "negative", "food": "negative"}
1969                              {"service": "negative"}
1970    {"cleanliness": "positive", "service": "positi...
Name: aspect_sentiments, Length: 1971, dtype: object>

In [17]:
df_train['clean_review'] = df_train['review_text'].apply(normalize_arabic)

In [18]:
df_train['clean_review'] 

0                     لا يوجد الدفع بالبطاقه عند الاستلام
1       المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...
2       تجربه سييه سالتهم الاكل هياخد وقت قد ايه قالول...
3                                         احلي مكان فزايد
4       الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا ي...
                              ...                        
1966    بصراحه كنت بحبو جدا جدا والناس كانت محترمه بس ...
1967    المطعم قعدته حلوه وفي زحليقه ومرجيحه للاطفال ا...
1968    The worst experience at the HUB Place is dirty...
1969    الكوفي في فندق الهيلتون رمسيس تم اضافه سلطه ال...
1970    الشغل كتير حلو ونظيف والاستقبال جميل جدا وهي ك...
Name: clean_review, Length: 1971, dtype: object

In [19]:
df_train['full_text'] = "منصة: " + df_train['platform'] + " . " + df_train['clean_review']

In [20]:
df_train['platform'].value_counts()

platform
google_maps    1210
play_store      761
Name: count, dtype: int64

In [21]:
df_train = pd.get_dummies(df_train, columns=['platform'])

In [22]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1971 entries, 0 to 1970
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   review_id             1971 non-null   int64         
 1   review_text           1971 non-null   object        
 2   star_rating           1971 non-null   int64         
 3   date                  1971 non-null   object        
 4   business_name         1971 non-null   object        
 5   business_category     1971 non-null   object        
 6   aspects               1971 non-null   object        
 7   aspect_sentiments     1971 non-null   object        
 8   date_parsed           1606 non-null   datetime64[ns]
 9   year                  1971 non-null   float64       
 10  month                 1971 non-null   float64       
 11  day_of_week           1971 non-null   float64       
 12  clean_review          1971 non-null   object        
 13  full_text         

In [23]:
df_train['platform_google_maps']

0       False
1        True
2        True
3        True
4        True
        ...  
1966    False
1967     True
1968     True
1969     True
1970     True
Name: platform_google_maps, Length: 1971, dtype: bool

In [24]:
df_train['platform_play_store']

0        True
1       False
2       False
3       False
4       False
        ...  
1966     True
1967    False
1968    False
1969    False
1970    False
Name: platform_play_store, Length: 1971, dtype: bool

In [25]:
#boolean ti int
df_train['platform_google_maps'] = df_train['platform_google_maps'].astype(int)
df_train['platform_play_store'] = df_train['platform_play_store'].astype(int)
df_train['star_rating'] = df_train['star_rating'].astype(int)

print(df_train['platform_google_maps'].isnull().sum())
print(df_train['platform_play_store'].isnull().sum())
print(df_train['star_rating'].isnull().sum())


print(df_train['platform_google_maps'].value_counts())
print(df_train['platform_play_store'].value_counts())
print(df_train['star_rating'].value_counts())



0
0
0
platform_google_maps
1    1210
0     761
Name: count, dtype: int64
platform_play_store
0    1210
1     761
Name: count, dtype: int64
star_rating
5    987
1    661
3    146
4     99
2     78
Name: count, dtype: int64


In [26]:
import ast

df_train['aspects'] = df_train['aspects'].apply(ast.literal_eval)
df_train['aspect_sentiments'] = df_train['aspect_sentiments'].apply(ast.literal_eval)

In [27]:
all_aspects = set()
for aspects in df_train['aspects']:
    all_aspects.update(aspects)

all_aspects = sorted(list(all_aspects))

In [28]:
for aspect in all_aspects:
    df_train[f"aspect_{aspect}"] = df_train['aspects'].apply(
        lambda x: 1 if aspect in x else 0
    )

In [29]:
sentiment_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

for aspect in all_aspects:
    df_train[f"sent_{aspect}"] = df_train['aspect_sentiments'].apply(
        lambda x: sentiment_map.get(x.get(aspect, "neutral"), 0)
    )

In [30]:
X_text = df_train['full_text']
X_num = df_train[['year', 'month', 'day_of_week', 'star_rating']]
y_aspects = df_train[[f"aspect_{a}" for a in all_aspects]]
y_sentiments = df_train[[f"sent_{a}" for a in all_aspects]]

In [31]:
X_text

0       منصة: play_store . لا يوجد الدفع بالبطاقه عند ...
1       منصة: google_maps . المكان نضيف وجميل وقعدته ت...
2       منصة: google_maps . تجربه سييه سالتهم الاكل هي...
3                     منصة: google_maps . احلي مكان فزايد
4       منصة: google_maps . الفطير حلو جدا الاحجام تحف...
                              ...                        
1966    منصة: play_store . بصراحه كنت بحبو جدا جدا وال...
1967    منصة: google_maps . المطعم قعدته حلوه وفي زحلي...
1968    منصة: google_maps . The worst experience at th...
1969    منصة: google_maps . الكوفي في فندق الهيلتون رم...
1970    منصة: google_maps . الشغل كتير حلو ونظيف والاس...
Name: full_text, Length: 1971, dtype: object

In [32]:
X_num

,year,month,day_of_week,star_rating
0,2026.0,3.0,6.0,3
1,2026.0,4.0,2.0,5
2,2026.0,3.0,2.0,1
3,2026.0,3.0,2.0,5
4,2025.0,4.0,3.0,4
...,...,...,...,...
1966,2026.0,3.0,1.0,1
1967,2026.0,1.0,5.0,3
1968,2024.0,4.0,2.0,1
1969,2022.0,4.0,0.0,1


In [33]:
y_aspects

,aspect_ambiance,aspect_app_experience,aspect_cleanliness,aspect_delivery,aspect_food,aspect_general,aspect_none,aspect_price,aspect_service
0,0,1,0,1,0,0,0,0,0
1,1,0,1,0,0,0,0,0,1
2,0,0,0,1,1,0,0,0,1
3,0,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...
1966,0,1,0,1,0,0,0,1,1
1967,1,0,0,0,1,0,0,1,1
1968,0,0,1,0,1,0,0,0,0
1969,0,0,0,0,0,0,0,0,1


In [34]:
y_sentiments

,sent_ambiance,sent_app_experience,sent_cleanliness,sent_delivery,sent_food,sent_general,sent_none,sent_price,sent_service
0,0,-1,0,-1,0,0,0,0,0
1,1,0,1,0,0,0,0,0,1
2,0,0,0,-1,0,0,0,0,-1
3,0,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...
1966,0,-1,0,-1,0,0,0,-1,1
1967,1,0,0,0,0,0,0,-1,-1
1968,0,0,-1,0,-1,0,0,0,0
1969,0,0,0,0,0,0,0,0,-1


******Modeling******

In [35]:
!pip install transformers datasets

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [36]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/aubmindlab/bert-base-arabertv02/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/aubmindlab/bert-base-arabertv02/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'aubmindlab/bert-base-arabertv02'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'aubmindlab/bert-base-arabertv02' is the correct path to a directory containing a config.json file

In [37]:
from transformers import AutoTokenizer, AutoModel

model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

tokenizer.save_pretrained("arabert_tokenizer")
model.save_pretrained("arabert_model")

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/aubmindlab/bert-base-arabertv02/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/aubmindlab/bert-base-arabertv02/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'aubmindlab/bert-base-arabertv02'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'aubmindlab/bert-base-arabertv02' is the correct path to a directory containing a config.json file

In [38]:
from arabert.preprocess import ArabertPreprocessor

model_name="bert-base-arabertv2"
arabert_prep = ArabertPreprocessor(model_name=model_name)

ModuleNotFoundError: No module named 'arabert'

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("./arabert_tokenizer")
model = AutoModel.from_pretrained("./arabert_model")

In [ ]:
all_aspects = sorted(set(a for row in df_train['aspects'] for a in row))
num_aspects = len(all_aspects)

aspect2idx = {a:i for i,a in enumerate(all_aspects)}

In [ ]:
def encode_labels(row):
    # multi-label aspects
    aspect_vec = [0]*num_aspects
    sentiment_vec = [0]*num_aspects  # -1,0,1 → later shift to 0,1,2

    for aspect in row['aspects']:
        idx = aspect2idx[aspect]
        aspect_vec[idx] = 1
        
        sent = row['aspect_sentiments'].get(aspect, "neutral")
        sentiment_map = {"negative":0, "neutral":1, "positive":2}
        sentiment_vec[idx] = sentiment_map[sent]

    return aspect_vec, sentiment_vec

df_train[['aspect_vec','sentiment_vec']] = df_train.apply(
    lambda row: pd.Series(encode_labels(row)), axis=1
)

In [ ]:
class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['full_text'].tolist()
        self.aspect_labels = df['aspect_vec'].tolist()
        self.sentiment_labels = df['sentiment_vec'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'aspect_labels': torch.tensor(self.aspect_labels[idx], dtype=torch.float),
            'sentiment_labels': torch.tensor(self.sentiment_labels[idx], dtype=torch.long)
        }

In [ ]:
class ABSAModel(nn.Module):
    def __init__(self, model_name, num_aspects):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.bert.config.hidden_size
        
        # aspect detection
        self.aspect_head = nn.Linear(hidden_size, num_aspects)
        
        # sentiment classification (per aspect)
        self.sentiment_head = nn.Linear(hidden_size, num_aspects * 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        cls = outputs.last_hidden_state[:, 0, :]  # CLS token
        
        aspect_logits = self.aspect_head(cls)
        sentiment_logits = self.sentiment_head(cls)
        
        sentiment_logits = sentiment_logits.view(-1, num_aspects, 3)
        
        return aspect_logits, sentiment_logits

In [ ]:
dataset = ABSADataset(df_train, tokenizer)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ABSAModel(MODEL_NAME, num_aspects).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

aspect_loss_fn = nn.BCEWithLogitsLoss()
sentiment_loss_fn = nn.CrossEntropyLoss()

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        aspect_labels = batch['aspect_labels'].to(device)
        sentiment_labels = batch['sentiment_labels'].to(device)

        optimizer.zero_grad()

        aspect_logits, sentiment_logits = model(input_ids, attention_mask)

        # aspect loss
        loss_aspect = aspect_loss_fn(aspect_logits, aspect_labels)

        # sentiment loss (flatten)
        loss_sentiment = sentiment_loss_fn(
            sentiment_logits.view(-1, 3),
            sentiment_labels.view(-1)
        )

        loss = loss_aspect + loss_sentiment
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

In [ ]:
def predict(text):
    model.eval()

    encoding = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        aspect_logits, sentiment_logits = model(input_ids, attention_mask)

    aspect_probs = torch.sigmoid(aspect_logits).cpu().numpy()[0]
    sentiment_preds = torch.argmax(sentiment_logits, dim=2).cpu().numpy()[0]

    result = {}

    for i, aspect in enumerate(all_aspects):
        if aspect_probs[i] > 0.5:
            sentiment_map = {0:"negative",1:"neutral",2:"positive"}
            result[aspect] = sentiment_map[sentiment_preds[i]]

    return result

In [ ]:
predict("منصة: google_maps . الاكل وحش والخدمة سيئة")